In [ ]:
# 그냥 이 셀은 돌리기
!pip install yfinance


In [ ]:
import yfinance as yf

def download_stock_data(tickers, start_date, end_date):
    """
    tickers: ['AAPL', 'GOOG', 'TSLA'] 와 같이 리스트 형태의 티커 목록
    start_date, end_date: 'YYYY-MM-DD' 형식의 날짜 문자열
    """

    # 여러 종목 한 번에 다운로드(group_by='ticker' 설정 주의)
    data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker')

    # 멀티 인덱스(MultiIndex) 형식으로 다운로드되므로, 티커별로 데이터 추출
    for ticker in tickers:
        # 혹시 종목 정보가 전혀 없거나 결측치만 있을 경우 dropna로 제거
        try:
            ticker_data = data[ticker].dropna(how='all')
        except KeyError:
            print(f"[경고] {ticker} 데이터가 존재하지 않습니다. 스킵합니다.")
            continue

        # 파일명 지정 (예: 'AAPL_2021-01-01_2021-12-31.csv')
        filename = f"{ticker}_{start_date}_{end_date}.csv"

        # CSV 파일 저장 (한글 인코딩을 위해 utf-8-sig 사용)
        ticker_data.to_csv(filename, encoding='utf-8-sig')
        print(f"[완료] {ticker} 데이터를 '{filename}' 파일로 저장했습니다.")


# 예: 사용자가 원하는 티커, 날짜를 직접 지정
my_tickers = ["TSLA"]
start_date_input = "2025-01-01"
end_date_input = "2025-05-28"

download_stock_data(my_tickers, start_date_input, end_date_input)


<ipython-input-4-456405351>:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker')
[*********************100%***********************]  1 of 1 completed

[완료] TSLA 데이터를 'TSLA_2025-01-01_2025-05-28.csv' 파일로 저장했습니다.


In [ ]:
## 아래 코드는 이용하지 않습니다 ##

import yfinance as yf
import pandas as pd

# ======================================================================
# 1) 주가 데이터 불러오기
# ======================================================================
# 티커 리스트 반영 완료!

tickers = [          # ← 여기에 티커 리스트를 붙여넣으세요
    "300750.SZ",     # 예: CATL
    "1211.HK",       # BYD
    "373220.KS"      # LG Energy Solution
]

# 가져올 기간 설정(예: 2010년 1월 1일 ~ 2025년 4월 14일)
start_date = "2023-01-01"
end_date = "2024-04-14"

# 주가 다운로드
price_data = yf.download(tickers, start=start_date, end=end_date)

# 가져온 주가 데이터를 엑셀로 저장하기
price_data.to_excel("price_data.xlsx")
print("주가 데이터가 'price_data.xlsx'로 저장되었습니다.")

# ======================================================================
# 2) 재무제표(재무상태표, 손익계산서) 불러오기
# ======================================================================
fin_statements = {}

for ticker in tickers:
    # 티커 객체 생성
    tk = yf.Ticker(ticker)

    # 재무상태표(분기, 연간), 손익계산서(분기, 연간) 불러오기
    q_balance = tk.quarterly_balance_sheet
    a_balance = tk.balance_sheet
    q_income = tk.quarterly_financials
    a_income = tk.financials

    # Dictionary에 저장
    fin_statements[ticker] = {
        "분기_재무상태표": q_balance,
        "연간_재무상태표": a_balance,
        "분기_손익계산서": q_income,
        "연간_손익계산서": a_income
    }

# ======================================================================
# 3) 불러온 재무제표를 Excel 파일에 여러 시트로 저장
# ======================================================================
# 여러 티커와 재무제표 유형이 있으니, ExcelWriter를 이용해 한 파일에 저장할 수 있음
with pd.ExcelWriter("financial_statements.xlsx") as writer:
    for ticker in fin_statements:
        # 티커별로 분기/연간 재무제표 시트에 기록
        # sheet_name이 31자 제한이 있으므로, 너무 길지 않게 작성
        fin_statements[ticker]["분기_재무상태표"].to_excel(writer, sheet_name=f"{ticker}_Q_BS")
        fin_statements[ticker]["연간_재무상태표"].to_excel(writer, sheet_name=f"{ticker}_A_BS")
        fin_statements[ticker]["분기_손익계산서"].to_excel(writer, sheet_name=f"{ticker}_Q_IS")
        fin_statements[ticker]["연간_손익계산서"].to_excel(writer, sheet_name=f"{ticker}_A_IS")

print("재무제표가 'financial_statements.xlsx' 파일로 저장되었습니다.")


<ipython-input-5-2393705923>:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  price_data = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  3 of 3 completed


주가 데이터가 'price_data.xlsx'로 저장되었습니다.
재무제표가 'financial_statements.xlsx' 파일로 저장되었습니다.


In [ ]:
# ────────────────────────────────────────────────────────────────────────
# 1) 필요 라이브러리 설치
# ────────────────────────────────────────────────────────────────────────
!pip install --upgrade yfinance pandas openpyxl --quiet

import yfinance as yf
import pandas as pd
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────
# 2) 사용자 설정
# ────────────────────────────────────────────────────────────────────────
tickers = [          # ← 여기에 티커 리스트를 붙여넣으세요
    "300750.SZ",     # 예: CATL
    "1211.HK",       # BYD
    "373220.KS"      # LG Energy Solution
]

start_date = "2015-01-01"   # 조회 시작일 (YYYY-MM-DD)
end_date   = "2025-04-28"   # 조회 종료일

# ────────────────────────────────────────────────────────────────────────
# 3) 저장 경로 생성
# ────────────────────────────────────────────────────────────────────────
out_dir = Path("/content/data")   # Google Drive 연동 시 경로만 바꿔주세요
out_dir.mkdir(parents=True, exist_ok=True)

# ────────────────────────────────────────────────────────────────────────
# 4) 주가 다운로드
# ────────────────────────────────────────────────────────────────────────
for tkr in tickers:
    print(f"▶ {tkr} : 주가 다운로드 중…")
    price_df = yf.download(
        tkr,
        start=start_date,
        end=end_date,
        auto_adjust=False,   # 필요하면 True로 변경 (배당/분할 반영)
        progress=False
    )
    price_df.to_csv(out_dir / f"price_{tkr.replace('.','_')}.csv")

# ────────────────────────────────────────────────────────────────────────
# 5) 연간 재무제표(최장 기간) 다운로드
# ────────────────────────────────────────────────────────────────────────
def fetch_financials(ticker: str):
    tk = yf.Ticker(ticker)
    # yfinance는 연간 기준으로 보통 최근 4~5년치만 제공합니다.
    bs  = tk.balance_sheet.T      # 재무상태표
    inc = tk.financials.T         # 손익계산서
    cf  = tk.cashflow.T           # 현금흐름표
    return bs, inc, cf

for tkr in tickers:
    print(f"▶ {tkr} : 재무제표 다운로드 중…")
    bs, inc, cf = fetch_financials(tkr)

    # 엑셀 통합 파일로 저장 (시트별)
    with pd.ExcelWriter(out_dir / f"fs_{tkr.replace('.','_')}.xlsx",
                        engine="openpyxl") as writer:
        bs.to_excel(writer, sheet_name="BalanceSheet")
        inc.to_excel(writer, sheet_name="IncomeStatement")
        cf.to_excel(writer, sheet_name="CashFlow")

print(f"\n✅ 모든 파일이 {out_dir} 경로에 저장되었습니다.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.4/118.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 34.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.0 which is incompatible.
cudf-cu12 25.2.1 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.0 which is incompatible.
dask-cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.0 which is incompatible.
▶ 300750.SZ : 주가 다운로드 중…
▶ 1211.HK : 주가 다운로드 중…
▶ 373220.KS : 주가 다운로드 중…
▶ 300750.SZ : 재무제표 다운로드 중…
▶ 1211.HK : 재무제표 다운로드 중…
▶ 373220.KS : 재무제표 다운로드 중…

✅ 모든 파일이 /content/data 경로에 저장되었습니다.
